In [2]:
import torch
import nemo.collections.asr as nemo_asr

d:\FineTuning\.myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.


In [2]:
import json
import soundfile as sf
import torch
from omegaconf import DictConfig
import nemo.collections.asr as nemo_asr

torch.set_num_threads(4)

model = nemo_asr.models.ASRModel.restore_from(
    "D:/FineTuning/final_after_150steps.nemo",
    map_location="cpu",
)
model.eval()

audio_path = r"D:\FineTuning\tuning_run\tuning_audio\196.wav"
duration = sf.info(audio_path).duration

manifest_path = "D:/FineTuning/manifest_infer.json"
with open(manifest_path, "w") as f:
    entry = {"audio_filepath": audio_path, "duration": duration, "text": "", "lang": "ne-NP"}
    f.write(json.dumps(entry) + "\n")

dl_config = {
    'manifest_filepath': manifest_path,
    'sample_rate': model.preprocessor._sample_rate,
    'labels': model.joint.vocabulary,
    'batch_size': 1,
    'trim_silence': False,
    'shuffle': False,
    'num_workers': 0,
    'pin_memory': False,
    'use_lhotse': True,
    'use_bucketing': False,
    'drop_last': False,
    'initialize_prompt_feature': True,
    'prompt_dictionary': model.cfg.model_defaults.get('prompt_dictionary'),
    'num_prompts': model.cfg.model_defaults.get('num_prompts', 128),
    'subsampling_factor': model.cfg.get('subsampling_factor', 8),
    'default_lang': 'ne-NP',
    'default_prompt_mode': 'langID',   # force deterministic language-forced decode, skip the unified coin-flip
    'window_stride': model.cfg.preprocessor.get('window_stride', 0.01),
}

dl = model._setup_dataloader_from_config(config=DictConfig(dl_config))

with torch.no_grad():
    for batch in dl:
        signal, signal_len, transcript, transcript_len, prompt_indices = batch
        encoded, encoded_len = model.forward(
            input_signal=signal, input_signal_length=signal_len, prompt_indices=prompt_indices
        )
        best_hyp = model.decoding.rnnt_decoder_predictions_tensor(
            encoder_output=encoded, encoded_lengths=encoded_len, return_hypotheses=False
        )
        print(best_hyp)

d:\FineTuning\.myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.


[NeMo I 2026-08-04 13:11:37 mixins:194] Tokenizer SentencePieceTokenizer initialized with 13087 tokens


[NeMo W 2026-08-04 13:11:41 modelPT:287] You tried to register an artifact under config key=tokenizer.model_path but an artifact for it has already been registered.
[NeMo W 2026-08-04 13:11:41 modelPT:287] You tried to register an artifact under config key=tokenizer.vocab_path but an artifact for it has already been registered.


[NeMo I 2026-08-04 13:11:41 mixins:194] Tokenizer SentencePieceTokenizer initialized with 13087 tokens


[NeMo W 2026-08-04 13:11:45 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /root/NemoASR/pilot_run/pilot_train_manifest.json
    sample_rate: 16000
    use_lhotse: true
    shard_manifests: true
    batch_duration: 8
    quadratic_duration: 15
    num_buckets: 30
    shuffle: true
    num_workers: 2
    pin_memory: true
    max_duration: 20
    min_duration: 0.1
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    slice_length: 100
    bucketing_strategy: fully_randomized
    bucketing_batch_size: null
    bucket_buffer_size: 10000
    shuffle_buffer_size: 10000
    prompt_field: target_lang
    prompt_dictionary:
      en-US: 0
      en: 0
      en-GB: 1
      enGB: 1
      es-ES: 2
      esES: 2
      es-US: 3
      es: 3
      zh-CN: 4
      zh-ZH: 4
      zh-TW: 5
      hi-IN: 6
   

[NeMo I 2026-08-04 13:11:50 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.005, 'clamp': -1.0}
[NeMo I 2026-08-04 13:11:50 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.005, 'clamp': -1.0}
[NeMo I 2026-08-04 13:11:50 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.005, 'clamp': -1.0}
[NeMo I 2026-08-04 13:11:50 rnnt_bpe_models_prompt:146] Model with prompt feature has been initialized (RNNT-only)
[NeMo I 2026-08-04 13:11:52 save_restore_connector:287] Model EncDecRNNTBPEModelWithPrompt was successfully restored from D:\FineTuning\final_after_150steps.nemo.
[NeMo I 2026-08-04 13:11:53 audio_to_text_lhotse_prompt_index:86] LhotseSpeechToTextBpeDatasetWithPromptIndex: default_prompt_mode=langID, unified_auto_ratio=0.5
[NeMo I 2026-08-04 13:11:53 rnnt_bpe_models_prompt:181] Setting up Lhotse dataset with prompt index supp

[NeMo W 2026-08-04 13:11:53 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: default_prompt_mode,initialize_prompt_feature,default_lang,window_stride,subsampling_factor,labels,num_prompts,trim_silence,prompt_dictionary
[NeMo W 2026-08-04 13:11:53 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


[NeMo I 2026-08-04 13:11:53 dataloader:659] Creating a Lhotse DynamicCutSampler (bucketing is disabled, (max_batch_duration=None max_batch_size=1)
[Hypothesis(score=-2.995201826095581, y_sequence=tensor([   2, 3259, 3250, 3332, 3317,    2, 3324, 3370, 3248, 3326,    2, 3327,
        3306, 3249, 3264,    2, 3327, 3280, 3320, 3312, 3363,    2, 3324, 3370,
        3248,    2, 3262, 3269, 3250, 3261, 3252]), text='पनौती यात्रामा भैरव भद्रकाली यात्रा हुन्स', dec_out=None, dec_state=None, timestamp=[], alignments=None, frame_confidence=None, token_confidence=None, word_confidence=None, length=0, y=None, lm_state=None, lm_scores=None, ngram_lm_state=None, tokens=None, last_token=None, token_duration=None, last_frame=None, biasing_cfg=None, non_blank_step_confidence_precomputed=None, xatt_scores=None)]
